# 04.05.04 PaddleOCR Fine-Tuning on OFPHIL (PP-OCRv6 SVTR Recognition)

**Phase 05 — Document Processing & OCR (OCR Model Benchmarks)**  
**Thesis:** A Multimodal Document Processing and Tampering Verification Pipeline for Patient Charity Assistance

### Research Objectives
1. Audit and separate the pretrained **PP-OCRv6_medium_det (DBNet)** text detector (inference-only / not fine-tuned in this notebook) from the **PP-OCRv6_medium_rec (LCNetV4 + EncoderWithLightSVTR)** text recognition model.
2. Verify pretrained recognition checkpoint trainability (`PirTranslatedLayer`), parameter gradient flow, character lexicon alignment (18,710 CTC classes), and CTC sequence constraints (`L + R <= T`).
3. Implement a complete, reproducible PP-OCRv6 SVTR text recognition fine-tuning pipeline on OFPHIL supporting document crops using `CTCLoss`, `AdamW`, and `CosineAnnealingDecay`.
4. Record per-epoch Training Loss, Validation Loss, Character Accuracy (%), CER, WER, Normalized Edit Distance (`1-NED`), Exact Match Accuracy (%), Learning Rate, Epoch Duration, and GPU VRAM Usage, with safe checkpoint management and resume support.
5. Evaluate the best validation checkpoint strictly once on the untouched held-out test partition (`test_manifest_v2.csv`) without data leakage, and render structured pandas summary tables and recorded Matplotlib curves.


### Inputs, Outputs & Strict Data Governance
- **Train Manifest (Fitting Only):** `data/interim/document_processing/splits/train_manifest_v2.csv`
- **Validation Manifest (Monitoring & Best Checkpoint Selection):** `data/interim/document_processing/splits/val_manifest_v2.csv`
- **Untouched Test Manifest (Final Evaluation Only — Zero Leakage):** `data/interim/document_processing/splits/test_manifest_v2.csv`
- **Pretrained Recognition Checkpoint:** `~/.paddlex/official_models/PP-OCRv6_medium_rec` (`LCNetV4 + EncoderWithLightSVTR`, CTC head)
- **Pretrained Detection Checkpoint (Audit Only):** `~/.paddlex/official_models/PP-OCRv6_medium_det` (`DBNet`, Pretrained / Inference-Only)
- **Checkpoint Directory:** `models/trained/ocr_finetuned/paddleocr/` (`best_accuracy/` and `last_checkpoint/`)
- **Configuration Artifact:** `models/trained/ocr_finetuned/paddleocr/paddleocr_rec_finetuning_config.yaml`
- **Training History Log:** `models/trained/ocr_finetuned/paddleocr/paddleocr_training_history.csv`
- **Final Evaluation Report & Predictions:** `data/interim/document_processing/model_predictions_and_benchmarks/paddleocr_finetuned/paddleocr_finetuned_benchmark_report.json` and `paddleocr_finetuned_test_predictions.csv`


### 1. Environment, Windows DLL Preload & Hardware Diagnostics


In [ ]:
import os
import sys
import gc
import re
import ast
import time
import math
import json
import yaml
import random
import shutil
import warnings
import unicodedata
from pathlib import Path

# Preserve Windows DLL search path setup for cuDNN 9 / shm.dll / CUDA runtime dependencies
# Note: Add torch/lib to DLL search path without importing torch when using legacy Paddle 2.6,
# or preload cleanly when running on PaddlePaddle >= 3.0 PIR kernels (e.g., venv312 / thesis_gpu_py312).
_torch_lib_candidates = [
    Path(sys.prefix) / 'Lib' / 'site-packages' / 'torch' / 'lib',
]
_REGISTERED_DLL_HANDLES = []
for _tlib in _torch_lib_candidates:
    if _tlib.exists():
        if hasattr(os, 'add_dll_directory'):
            try:
                _REGISTERED_DLL_HANDLES.append(os.add_dll_directory(str(_tlib)))
            except OSError:
                pass
        os.environ['PATH'] = str(_tlib) + os.pathsep + os.environ.get('PATH', '')

# In PaddlePaddle >= 3.0 PIR kernels on Windows, preloading torch loads cuDNN 9 / shm.dll cleanly;
# skip importing torch on legacy Paddle 2.x to avoid duplicate pybind11 _gpuDeviceProperties registration.
try:
    import importlib.metadata as _im
    _pd_ver = _im.version('paddlepaddle-gpu')
except Exception:
    _pd_ver = '0'
if int(str(_pd_ver).split('.')[0] or 0) >= 3:
    try:
        import torch as _torch_dll_preload  # noqa: F401
    except Exception:
        pass

warnings.filterwarnings('ignore')
os.environ.setdefault('GLOG_minloglevel', '3')
os.environ.setdefault('PADDLE_PDX_LOGGING_LEVEL', 'ERROR')

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

import paddle
import paddle.nn.functional as F

# Deterministic seeding for reproducibility
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
paddle.seed(RANDOM_SEED)

def _find_root(p: Path = Path.cwd()) -> Path:
    for parent in [p.resolve(), *p.resolve().parents]:
        if (parent / 'data').exists() and (parent / 'experiments').exists():
            return parent
    return p.resolve()

PROJECT_ROOT = _find_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.document_processing import DocumentProcessingPaths
paths = DocumentProcessingPaths(PROJECT_ROOT)

# Sibling repository raw image fallback (read-only) when raw captures are stored in the primary thesis workspace
SIBLING_RAW_DOCS_DIR = PROJECT_ROOT.parent / 'dss-charity-eligibity-classification' / 'data' / 'raw' / 'supporting document' / 'supporting_docu'

TRAIN_MANIFEST = paths.splits_dir / 'train_manifest_v2.csv'
VAL_MANIFEST = paths.splits_dir / 'val_manifest_v2.csv'
TEST_MANIFEST = paths.splits_dir / 'test_manifest_v2.csv'

OUTPUT_DIR = paths.root / 'models' / 'trained' / 'ocr_finetuned' / 'paddleocr'
BEST_CKPT_DIR = OUTPUT_DIR / 'best_accuracy'
LAST_CKPT_DIR = OUTPUT_DIR / 'last_checkpoint'
HISTORY_CSV_PATH = OUTPUT_DIR / 'paddleocr_training_history.csv'
TRAINING_STATE_JSON = OUTPUT_DIR / 'training_state.json'
CONFIG_YAML_PATH = OUTPUT_DIR / 'paddleocr_rec_finetuning_config.yaml'

BENCHMARK_OUT_DIR = paths.model_benchmarks_dir / 'paddleocr_finetuned'
FINAL_REPORT_PATH = BENCHMARK_OUT_DIR / 'paddleocr_finetuned_benchmark_report.json'
TEST_PREDICTIONS_CSV = BENCHMARK_OUT_DIR / 'paddleocr_finetuned_test_predictions.csv'

for _d in (OUTPUT_DIR, BEST_CKPT_DIR, LAST_CKPT_DIR, BENCHMARK_OUT_DIR):
    _d.mkdir(parents=True, exist_ok=True)

# Automatic CUDA / CPU device detection via PaddlePaddle
HAS_CUDA = bool(paddle.is_compiled_with_cuda() and paddle.device.cuda.device_count() > 0)
ACTIVE_DEVICE = 'gpu:0' if HAS_CUDA else 'cpu'
paddle.set_device(ACTIVE_DEVICE)
GPU_NAME = paddle.device.cuda.get_device_name(0) if HAS_CUDA else 'CPU'

def get_gpu_vram_mb() -> float:
    if HAS_CUDA:
        try:
            return float(paddle.device.cuda.max_memory_allocated(0) / (1024 ** 2))
        except Exception:
            try:
                return float(paddle.device.cuda.memory_allocated(0) / (1024 ** 2))
            except Exception:
                return 0.0
    return 0.0

# Track pipeline status truthfully across notebook cells
PIPELINE_RUNTIME_STATUS = 'PENDING MANUAL EXECUTION'
PIPELINE_ERROR_DETAIL = None

print(f"Project Root        : {paths.root}")
print(f"Python Executable   : {sys.executable}")
print(f"PaddlePaddle Version: {paddle.__version__} (CUDA Compiled: {paddle.is_compiled_with_cuda()})")
print(f"Active Device       : {ACTIVE_DEVICE.upper()} ({GPU_NAME})")
print(f"Output Checkpoint   : {OUTPUT_DIR}")
print(f"Manifests Present   : Train={TRAIN_MANIFEST.exists()} | Val={VAL_MANIFEST.exists()} | Test={TEST_MANIFEST.exists()}")


### 2. Architectural Audit (DBNet vs. SVTR) & Fine-Tuning Hyperparameters

**Architectural Separation Notice:**
- **Text Detection (`PP-OCRv6_medium_det` — DBNet + RepLKFPN):** Pretrained / **Inference-Only** in this notebook. No detection training loop is executed here, and detection metrics are never merged into recognition accuracy.
- **Text Recognition (`PP-OCRv6_medium_rec` — LCNetV4 + EncoderWithLightSVTR):** **Trainable Target** in this notebook. In PaddleOCR 3.x / PaddleX, `PP-OCRv6_medium_rec` is serialized in Paddle PIR format (`inference.json` + `inference.pdiparams` with `"trainable": true`), requiring **PaddlePaddle >= 3.0** (`paddle.jit.load` -> `PirTranslatedLayer`) for native gradient backpropagation across its 155 trainable parameter tensors.


In [ ]:
# Locate Official Pretrained PP-OCRv6 Checkpoints (DBNet Detection vs. SVTR Recognition)
PADDLEX_MODELS_ROOT = Path.home() / '.paddlex' / 'official_models'
PRETRAINED_DET_DIR = PADDLEX_MODELS_ROOT / 'PP-OCRv6_medium_det'
PRETRAINED_REC_DIR = PADDLEX_MODELS_ROOT / 'PP-OCRv6_medium_rec'

def audit_pretrained_ocr_components(det_dir: Path, rec_dir: Path) -> pd.DataFrame:
    rows = []
    # 1. Audit DBNet Detection Checkpoint (Inference-Only)
    det_yml = det_dir / 'inference.yml'
    det_json = det_dir / 'inference.json'
    det_params = det_dir / 'inference.pdiparams'
    det_post = 'Unknown'
    if det_yml.exists():
        with open(det_yml, 'r', encoding='utf-8') as f:
            d_cfg = yaml.safe_load(f) or {}
        det_post = d_cfg.get('PostProcess', {}).get('name', 'DBPostProcess')
    rows.append({
        'Component': 'Text Detection',
        'Model Identifier': 'PP-OCRv6_medium_det',
        'Architecture': f'LCNetV4 + RepLKFPN + {det_post} (DBNet)',
        'Checkpoint Path': str(det_dir),
        'Weights Present': bool(det_json.exists() and det_params.exists()),
        'Weights Size (MB)': round(det_params.stat().st_size / (1024 ** 2), 2) if det_params.exists() else 0.0,
        'Notebook Role / Status': 'PRETRAINED / INFERENCE-ONLY (Not Fine-Tuned Here)',
    })

    # 2. Audit SVTR Recognition Checkpoint (Fine-Tuning Target)
    rec_yml = rec_dir / 'inference.yml'
    rec_json = rec_dir / 'inference.json'
    rec_params = rec_dir / 'inference.pdiparams'
    pir_trainable_flag = False
    rec_vocab_len = 0
    if rec_yml.exists():
        with open(rec_yml, 'r', encoding='utf-8') as f:
            r_cfg = yaml.safe_load(f) or {}
        rec_vocab_len = len(r_cfg.get('PostProcess', {}).get('character_dict', []))
    if rec_json.exists():
        with open(rec_json, 'r', encoding='utf-8') as f:
            head_chunk = f.read(512)
        pir_trainable_flag = '"trainable":true' in head_chunk.replace(' ', '')
    rows.append({
        'Component': 'Text Recognition',
        'Model Identifier': 'PP-OCRv6_medium_rec',
        'Architecture': f'LCNetV4 + EncoderWithLightSVTR + CTCHead ({rec_vocab_len + 2:,} classes)',
        'Checkpoint Path': str(rec_dir),
        'Weights Present': bool(rec_json.exists() and rec_params.exists()),
        'Weights Size (MB)': round(rec_params.stat().st_size / (1024 ** 2), 2) if rec_params.exists() else 0.0,
        'Notebook Role / Status': f'FINE-TUNING TARGET (PIR Trainable={pir_trainable_flag})',
    })
    return pd.DataFrame(rows)

df_component_audit = audit_pretrained_ocr_components(PRETRAINED_DET_DIR, PRETRAINED_REC_DIR)
print("=== DBNet DETECTION VS. SVTR RECOGNITION ARCHITECTURAL AUDIT ===")
display(df_component_audit)

# Explicit PaddleOCR Recognition Fine-Tuning Hyperparameters
# Tuned for NVIDIA RTX 3050 Laptop GPU (4 GB VRAM) and limited system RAM:
# - img_height=48, img_width=640 -> stride-8 width yields T = 80 CTC time steps
# - batch_size=8 keeps peak GPU VRAM ~650-850 MB (safe under 4 GB VRAM budget)
# - learning_rate=5e-5 with AdamW & CosineAnnealingDecay prevents catastrophic forgetting of the 18,710-class head
PADDLEOCR_TRAINING_CONFIG = {
    'model_name': 'PP-OCRv6_medium_rec',
    'architecture': 'LCNetV4 + EncoderWithLightSVTR + CTC',
    'detection_model_name': 'PP-OCRv6_medium_det (DBNet - Pretrained / Inference-Only)',
    'pretrained_rec_dir': str(PRETRAINED_REC_DIR),
    'pretrained_det_dir': str(PRETRAINED_DET_DIR),
    'max_epochs': 10,
    'batch_size': 8,
    'eval_batch_size': 16,
    'learning_rate': 5e-5,
    'min_learning_rate': 1e-6,
    'weight_decay': 1e-4,
    'grad_clip_norm': 5.0,
    'lr_scheduler': 'CosineAnnealingDecay',
    'loss_function': 'CTCLoss(blank=0, reduction=mean)',
    'early_stopping_patience': 4,
    'selection_metric': 'val_cer (lower is better; tie-break on val_norm_ed)',
    'resume_from_checkpoint': True,
    'img_height': 48,
    'img_width': 640,
    'ctc_time_steps': 80,
    'max_label_length': 75,
    'num_workers': 0,
    'seed': RANDOM_SEED,
    'device': ACTIVE_DEVICE,
}

with open(CONFIG_YAML_PATH, 'w', encoding='utf-8') as f:
    yaml.safe_dump(PADDLEOCR_TRAINING_CONFIG, f, sort_keys=False, allow_unicode=True)

df_config_display = pd.DataFrame(
    [{'Parameter': k, 'Configured Value': str(v)} for k, v in PADDLEOCR_TRAINING_CONFIG.items()]
)
print(f"Saved fine-tuning configuration -> {CONFIG_YAML_PATH}")
display(df_config_display)


### 3. Metric Calculations: Levenshtein Edit Distance, Character Accuracy, CER, WER, 1-NED & Exact Match

Metric definitions enforced across validation and held-out test evaluation:
- **Edit Distance $d(r, h)$:** Exact dynamic programming Levenshtein distance (insertions + deletions + substitutions).
- **CER (Character Error Rate):** Uncapped sample ratio $d_{\text{char}}(r, h) / \max(1, |r|)$ averaged across samples (with corpus micro-CER also tracked in evaluation reports).
- **WER (Word Error Rate):** Uncapped whitespace-tokenized word Levenshtein ratio $d_{\text{word}}(r, h) / \max(1, |w_r|)$.
- **Normalized Edit Distance (`1 - NED`):** Standard ICDAR / PaddleOCR sequence similarity metric $1 - d_{\text{char}}(r, h) / \max(|r|, |h|, 1) \in [0, 1]$, guaranteed bounded even when $|h| > |r|$.
- **Character Accuracy (%):** Computed directly from non-negative character alignment accuracy $\max(0, |r| - d_{\text{char}}(r, h)) / \max(1, |r|) \times 100\%$ per sample (preventing negative accuracy distortion when insertions exceed reference length, rather than naively taking `1 - uncapped_CER`).
- **Exact Match Accuracy (%):** Percentage of crops where cleaned predicted text matches cleaned ground-truth text character-for-character (`pred == gt`).


In [ ]:
def clean_ocr_text(text) -> str:
    """Canonical NFC text sanitizer preserving case, digits, punctuation, and Filipino/medical glyphs."""
    if text is None or (isinstance(text, float) and math.isnan(text)):
        return ''
    t = str(text)
    if t.strip().lower() == 'nan':
        return ''
    # Standardize clinical shorthand c-macron ('with') consistently with OFPHIL annotations
    t = t.replace('c\u0304', 'with').replace('c̄', 'with')
    t = re.sub(r'[\r\n\t]+', ' ', t)
    t = re.sub(r'\s+', ' ', t).strip()
    return unicodedata.normalize('NFC', t)


def levenshtein_distance(seq1, seq2) -> int:
    """Space-optimized O(min(N, M)) exact Levenshtein edit distance for strings or token lists."""
    n, m = len(seq1), len(seq2)
    if n == 0:
        return m
    if m == 0:
        return n
    if n > m:
        seq1, seq2 = seq2, seq1
        n, m = m, n

    prev_row = list(range(n + 1))
    for i in range(1, m + 1):
        curr_row = [i] + [0] * n
        s2_item = seq2[i - 1]
        for j in range(1, n + 1):
            cost = 0 if seq1[j - 1] == s2_item else 1
            curr_row[j] = min(
                prev_row[j] + 1,       # insertion
                curr_row[j - 1] + 1,   # deletion
                prev_row[j - 1] + cost # substitution
            )
        prev_row = curr_row
    return int(prev_row[n])


def compute_sample_ocr_metrics(reference: str, hypothesis: str) -> dict:
    """Computes per-sample CER, WER, Character Accuracy (%), Normalized Edit Distance (1-NED), and Exact Match."""
    r = clean_ocr_text(reference)
    h = clean_ocr_text(hypothesis)

    if len(r) == 0:
        is_empty_match = (len(h) == 0)
        return {
            'char_dist': len(h),
            'ref_char_len': 0,
            'cer': 0.0 if is_empty_match else 1.0,
            'char_acc_pct': 100.0 if is_empty_match else 0.0,
            'norm_ed_1_minus_ned': 1.0 if is_empty_match else 0.0,
            'word_dist': 0 if is_empty_match else len(h.split()),
            'ref_word_len': 0,
            'wer': 0.0 if is_empty_match else 1.0,
            'exact_match_pct': 100.0 if is_empty_match else 0.0,
        }

    c_dist = levenshtein_distance(r, h)
    r_len = len(r)
    max_len = max(r_len, len(h), 1)

    cer = float(c_dist / r_len)
    # Bounded per-reference character recognition accuracy (avoids negative values when len(h) >> len(r))
    char_acc_pct = float(max(0.0, (r_len - c_dist) / r_len) * 100.0)
    # ICDAR Normalized Edit Distance similarity: 1 - d(r, h) / max(|r|, |h|)
    one_minus_ned = float(1.0 - (c_dist / max_len))

    r_words = r.split()
    h_words = h.split()
    w_dist = levenshtein_distance(r_words, h_words)
    rw_len = len(r_words)
    wer = float(w_dist / rw_len) if rw_len > 0 else (0.0 if len(h_words) == 0 else 1.0)

    exact_match_pct = 100.0 if (r == h) else 0.0

    return {
        'char_dist': c_dist,
        'ref_char_len': r_len,
        'cer': cer,
        'char_acc_pct': char_acc_pct,
        'norm_ed_1_minus_ned': one_minus_ned,
        'word_dist': w_dist,
        'ref_word_len': rw_len,
        'wer': wer,
        'exact_match_pct': exact_match_pct,
    }


def compute_cer(reference: str, hypothesis: str) -> float:
    return compute_sample_ocr_metrics(reference, hypothesis)['cer']


def compute_wer(reference: str, hypothesis: str) -> float:
    return compute_sample_ocr_metrics(reference, hypothesis)['wer']


def aggregate_ocr_metrics(references: list, hypotheses: list) -> dict:
    """Aggregates dataset-level OCR metrics across all evaluated crops."""
    if not references:
        return {
            'num_samples': 0,
            'char_acc_pct': 0.0,
            'cer': 0.0,
            'micro_cer': 0.0,
            'wer': 0.0,
            'micro_wer': 0.0,
            'norm_ed': 0.0,
            'exact_match_pct': 0.0,
        }
    per_sample = [compute_sample_ocr_metrics(r, h) for r, h in zip(references, hypotheses)]
    tot_c_dist = sum(m['char_dist'] for m in per_sample)
    tot_c_len = sum(m['ref_char_len'] for m in per_sample)
    tot_w_dist = sum(m['word_dist'] for m in per_sample)
    tot_w_len = sum(m['ref_word_len'] for m in per_sample)

    return {
        'num_samples': len(per_sample),
        'char_acc_pct': float(np.mean([m['char_acc_pct'] for m in per_sample])),
        'cer': float(np.mean([m['cer'] for m in per_sample])),
        'micro_cer': float(tot_c_dist / max(1, tot_c_len)),
        'wer': float(np.mean([m['wer'] for m in per_sample])),
        'micro_wer': float(tot_w_dist / max(1, tot_w_len)),
        'norm_ed': float(np.mean([m['norm_ed_1_minus_ned'] for m in per_sample])),
        'exact_match_pct': float(np.mean([m['exact_match_pct'] for m in per_sample])),
    }

# Quick deterministic self-check on metric edge cases
_chk_exact = compute_sample_ocr_metrics('PHACO WITH IOL OS', 'PHACO WITH IOL OS')
_chk_ins = compute_sample_ocr_metrics('51', '51000')
assert _chk_exact['cer'] == 0.0 and _chk_exact['exact_match_pct'] == 100.0 and _chk_exact['norm_ed_1_minus_ned'] == 1.0
assert _chk_ins['cer'] == 1.5 and _chk_ins['char_acc_pct'] == 0.0 and abs(_chk_ins['norm_ed_1_minus_ned'] - 0.4) < 1e-6
print("[OK] Validated Levenshtein CER, WER, Character Accuracy (%), 1-NED, and Exact Match (%) formulations.")


### 4. Character Vocabulary, CTC Length Validator, Dataset Loader & Trainable SVTR Builder

**Critical Technical Verifications Implemented Here:**
1. **Exact Vocabulary Mapping (18,710 classes):** Reconstructs `CTCLabelDecode` from `PP-OCRv6_medium_rec/inference.yml` where index `0` is `<blank>`, indices `1..18708` map `character_dict`, and index `18709` maps `' '` (`use_space_char=True`), matching the final linear projection `[192, 18710]`.
2. **CTC Sequence Feasibility (`L + R <= T`):** With input `[3, 48, 640]`, the SVTR neck emits `T = 80` time steps (`W // 8`). Because CTC inserts a mandatory blank between adjacent identical tokens (`R` repeats), `truncate_ctc_tokens()` enforces `len(tokens) + repeats <= 80`, preventing infinite/zeroed Warp-CTC gradients on long crops.
3. **Zero Data Leakage & Safe Image Resolution:** `train_manifest_v2.csv`, `val_manifest_v2.csv`, and `test_manifest_v2.csv` are verified for disjoint `document_id` and `crop_id` sets; `TEST_MANIFEST` is not instantiated until Section 6. Document images are resolved cleanly across V2 `capture/` directories and read-only sibling raw document folders, using deterministic LRU-free per-crop reading to preserve system RAM.
4. **Trainable Pretrained Model Verification:** `load_trainable_ppocrv6_rec()` verifies Paddle PIR support (`paddle.__version__ >= 3.0`), loads `PP-OCRv6_medium_rec/inference`, switches to `.train()`, confirms trainable parameters (`stop_gradient == False`), and validates output tensor dimensions `[B, T, 18710]` (post-softmax probabilities converted via `paddle.log(paddle.clip(out, min=1e-7))` for `paddle.nn.CTCLoss`).


In [ ]:
# 1. Load Official PP-OCRv6 Character Lexicon
def load_ppocrv6_lexicon(rec_model_dir: Path):
    yml_path = rec_model_dir / 'inference.yml'
    if not yml_path.exists():
        raise FileNotFoundError(
            f"Pretrained PP-OCRv6 recognition config not found at {yml_path}. "
            f"Ensure PP-OCRv6_medium_rec is cached under {rec_model_dir}."
        )
    with open(yml_path, 'r', encoding='utf-8') as f:
        rec_cfg = yaml.safe_load(f)

    post_cfg = rec_cfg.get('PostProcess', {})
    if post_cfg.get('name') != 'CTCLabelDecode':
        raise ValueError(f"Expected CTCLabelDecode post-processor, got: {post_cfg.get('name')}")

    char_dict = list(post_cfg['character_dict'])
    # Index 0 = CTC blank; 1..N = character_dict; N+1 = space character (' ')
    full_char_list = ['<blank>'] + char_dict + [' ']
    char2idx = {ch: idx + 1 for idx, ch in enumerate(char_dict)}
    space_idx = len(full_char_list) - 1
    char2idx[' '] = space_idx
    return full_char_list, char2idx, space_idx

FULL_CHAR_LIST, CHAR2IDX, SPACE_TOKEN_IDX = load_ppocrv6_lexicon(PRETRAINED_REC_DIR)
NUM_CTC_CLASSES = len(FULL_CHAR_LIST)
print(f"[LEXICON] Loaded PP-OCRv6 vocabulary: {NUM_CTC_CLASSES:,} total classes (Blank=0, Space={SPACE_TOKEN_IDX})")


# 2. Aspect-Ratio Preserving Image Normalizer (BGR [H, W, 3] -> CHW [3, 48, 640] in [-1, 1])
def preprocess_image_crop(
    img: np.ndarray,
    target_h: int = PADDLEOCR_TRAINING_CONFIG['img_height'],
    target_w: int = PADDLEOCR_TRAINING_CONFIG['img_width'],
) -> np.ndarray:
    if img is None or img.size == 0:
        return np.zeros((3, target_h, target_w), dtype=np.float32)
    if img.ndim == 2:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    elif img.shape[2] == 4:
        img = cv2.cvtColor(img, cv2.COLOR_BGRA2BGR)

    h, w = img.shape[:2]
    ratio = w / float(max(1, h))
    resized_w = min(target_w, max(1, int(math.ceil(target_h * ratio))))
    interp = cv2.INTER_AREA if resized_w < w else cv2.INTER_LINEAR
    resized_img = cv2.resize(img, (resized_w, target_h), interpolation=interp)

    norm_img = (resized_img.astype(np.float32) / 255.0 - 0.5) / 0.5
    padded_img = np.zeros((3, target_h, target_w), dtype=np.float32)
    padded_img[:, :, :resized_w] = norm_img.transpose(2, 0, 1)
    return padded_img


# 3. CTC Target Sequence Encoder & Length Validator (guarantees L + adjacent_repeats <= T)
def truncate_ctc_tokens(
    token_ids: list,
    max_time_steps: int = PADDLEOCR_TRAINING_CONFIG['ctc_time_steps'],
    max_label_len: int = PADDLEOCR_TRAINING_CONFIG['max_label_length'],
) -> list:
    safe_tokens = []
    required_steps = 0
    for tok in token_ids[:max_label_len]:
        extra = 2 if (safe_tokens and tok == safe_tokens[-1]) else 1
        if required_steps + extra > max_time_steps:
            break
        safe_tokens.append(tok)
        required_steps += extra
    return safe_tokens


def encode_text_for_ctc(
    text: str,
    char2idx: dict = CHAR2IDX,
    max_time_steps: int = PADDLEOCR_TRAINING_CONFIG['ctc_time_steps'],
    max_label_len: int = PADDLEOCR_TRAINING_CONFIG['max_label_length'],
) -> list:
    cleaned = clean_ocr_text(text)
    raw_ids = [char2idx[c] for c in cleaned if c in char2idx]
    return truncate_ctc_tokens(raw_ids, max_time_steps=max_time_steps, max_label_len=max_label_len)


def ctc_greedy_decode(probs_or_logits: np.ndarray, full_character_list: list = FULL_CHAR_LIST) -> list:
    """Standard CTC collapse-repeats-then-remove-blank greedy decoder."""
    best_ids = np.argmax(probs_or_logits, axis=-1)
    vocab_size = len(full_character_list)
    decoded_texts = []
    for seq in best_ids:
        if len(seq) == 0:
            decoded_texts.append('')
            continue
        keep = np.ones(len(seq), dtype=bool)
        keep[1:] = seq[1:] != seq[:-1]
        keep &= (seq != 0)
        chars = [full_character_list[int(tok)] for tok in seq[keep] if 0 < int(tok) < vocab_size]
        decoded_texts.append(clean_ocr_text(''.join(chars)))
    return decoded_texts


# 4. Deterministic Document Image Path Resolver
def build_document_image_index(doc_paths: DocumentProcessingPaths, sibling_raw_dir: Path) -> dict:
    index = {}
    search_roots = [
        doc_paths.raw_documents_dir,
        doc_paths.interim_doc_proc / 'preprocessed_docs',
    ]
    if sibling_raw_dir.exists():
        search_roots.append(sibling_raw_dir)
    for base in search_roots:
        if base and base.exists():
            for fpath in base.rglob('*'):
                if fpath.is_file() and fpath.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}:
                    index.setdefault(fpath.name, fpath)
    return index

IMAGE_FILENAME_INDEX = build_document_image_index(paths, SIBLING_RAW_DOCS_DIR)

def resolve_manifest_image_path(rel_path: str, image_filename: str = '') -> Path:
    # 1. Standard V2 resolver
    p_v2 = paths.resolve_sample_path(str(rel_path))
    if p_v2.exists():
        return p_v2
    # 2. Map physical_capture / scan_capture -> V2 unified capture/ folder
    norm_rel = str(rel_path).replace('\\', '/')
    for legacy_seg in ('/physical_capture/', '/scan_capture/'):
        if legacy_seg in norm_rel:
            p_cap = paths.root / norm_rel.replace(legacy_seg, '/capture/')
            if p_cap.exists():
                return p_cap
    # 3. Sibling thesis workspace raw directory (read-only)
    if SIBLING_RAW_DOCS_DIR.exists():
        p_sib = PROJECT_ROOT.parent / 'dss-charity-eligibity-classification' / norm_rel
        if p_sib.exists():
            return p_sib
    # 4. Exact filename index fallback
    fname = image_filename or Path(norm_rel).name
    if fname in IMAGE_FILENAME_INDEX:
        return IMAGE_FILENAME_INDEX[fname]
    return p_v2


def parse_bbox(bbox_val) -> list:
    if isinstance(bbox_val, (list, tuple)) and len(bbox_val) == 4:
        return [int(round(float(x))) for x in bbox_val]
    if isinstance(bbox_val, str):
        parsed = json.loads(bbox_val) if bbox_val.strip().startswith('[') else ast.literal_eval(bbox_val)
        return [int(round(float(x))) for x in parsed[:4]]
    raise ValueError(f"Invalid bounding_box format: {bbox_val!r}")


# 5. Memory-Safe OFPHIL Crop Dataset (reads crops on-demand without caching full-page scans in RAM)
class OFPHILRecognitionDataset(paddle.io.Dataset):
    def __init__(
        self,
        manifest_df: pd.DataFrame,
        split_name: str,
        char2idx: dict = CHAR2IDX,
        img_h: int = PADDLEOCR_TRAINING_CONFIG['img_height'],
        img_w: int = PADDLEOCR_TRAINING_CONFIG['img_width'],
        ctc_steps: int = PADDLEOCR_TRAINING_CONFIG['ctc_time_steps'],
        max_len: int = PADDLEOCR_TRAINING_CONFIG['max_label_length'],
    ):
        super().__init__()
        self.split_name = split_name
        self.char2idx = char2idx
        self.img_h = img_h
        self.img_w = img_w
        self.ctc_steps = ctc_steps
        self.max_len = max_len

        self.samples = []
        self.skipped_empty_or_oov = 0
        self.missing_image_files = 0
        self.truncated_for_ctc = 0

        for _, row in manifest_df.iterrows():
            gt_text = clean_ocr_text(row.get('ground_truth_text', ''))
            if not gt_text:
                self.skipped_empty_or_oov += 1
                continue

            raw_tokens = [self.char2idx[c] for c in gt_text if c in self.char2idx]
            encoded = truncate_ctc_tokens(raw_tokens, max_time_steps=self.ctc_steps, max_label_len=self.max_len)
            if not encoded:
                self.skipped_empty_or_oov += 1
                continue
            if len(encoded) < len(raw_tokens):
                self.truncated_for_ctc += 1

            img_fname = str(row.get('image_filename', Path(str(row['image_path'])).name))
            resolved_path = resolve_manifest_image_path(row['image_path'], img_fname)
            if not resolved_path.exists():
                self.missing_image_files += 1
                continue

            bbox = parse_bbox(row['bounding_box'])
            if bbox[2] <= 0 or bbox[3] <= 0:
                self.skipped_empty_or_oov += 1
                continue

            self.samples.append({
                'crop_id': str(row.get('crop_id', '')),
                'document_id': str(row.get('document_id', '')),
                'image_filename': img_fname,
                'img_path': str(resolved_path),
                'bbox': bbox,
                'ground_truth_text': gt_text,
                'encoded_label': encoded,
                'document_type': str(row.get('document_type', 'unknown')),
            })

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int):
        item = self.samples[idx]
        bx, by, bw, bh = item['bbox']
        full_img = cv2.imread(item['img_path'], cv2.IMREAD_COLOR)
        if full_img is None:
            crop = np.zeros((self.img_h, self.img_w, 3), dtype=np.uint8)
        else:
            ih, iw = full_img.shape[:2]
            x1 = max(0, min(bx, iw - 1))
            y1 = max(0, min(by, ih - 1))
            x2 = max(x1 + 1, min(bx + bw, iw))
            y2 = max(y1 + 1, min(by + bh, ih))
            crop = full_img[y1:y2, x1:x2]
            if crop.size == 0:
                crop = np.zeros((self.img_h, self.img_w, 3), dtype=np.uint8)

        tensor_img = preprocess_image_crop(crop, self.img_h, self.img_w)
        label_arr = np.array(item['encoded_label'], dtype=np.int32)
        return (
            tensor_img,
            label_arr,
            len(label_arr),
            item['ground_truth_text'],
            item['image_filename'],
            item['crop_id'],
        )


def ocr_ctc_collate_fn(batch):
    imgs, labels, lengths, gt_texts, filenames, crop_ids = zip(*batch)
    bsz = len(batch)
    max_lbl_len = max(lengths)
    padded_labels = np.zeros((bsz, max_lbl_len), dtype=np.int32)
    for i, seq in enumerate(labels):
        padded_labels[i, :len(seq)] = seq

    batch_imgs = paddle.to_tensor(np.stack(imgs, axis=0), dtype='float32')
    batch_labels = paddle.to_tensor(padded_labels, dtype='int32')
    batch_target_lens = paddle.to_tensor(np.array(lengths, dtype=np.int64), dtype='int64')
    return batch_imgs, batch_labels, batch_target_lens, list(gt_texts), list(filenames), list(crop_ids)


# 6. Trainable PP-OCRv6 Recognition Model Loader & Forward/CTC Adapter
def load_trainable_ppocrv6_rec(pretrained_dir: Path = PRETRAINED_REC_DIR, for_training: bool = True):
    """
    Loads PP-OCRv6_medium_rec PIR checkpoint via paddle.jit.load and verifies:
      1. PaddlePaddle >= 3.0 PIR runtime compatibility (inference.json + inference.pdiparams).
      2. Trainable parameter tensors exist (stop_gradient == False) when for_training=True.
      3. Forward output dimensions match [B, T, 18710] and normalize to probability simplex (or raw logits).
    """
    major_ver = int(str(paddle.__version__).split('.')[0])
    pir_json = pretrained_dir / 'inference.json'
    legacy_pdmodel = pretrained_dir / 'inference.pdmodel'

    if pir_json.exists() and not legacy_pdmodel.exists() and major_ver < 3:
        raise RuntimeError(
            f"PP-OCRv6_medium_rec uses Paddle PIR format ({pir_json.name}), which requires PaddlePaddle >= 3.0, "
            f"but current kernel runs PaddlePaddle {paddle.__version__} ({sys.executable}). "
            f"Switch Jupyter kernel to 'Python (venv312)' / 'Python 3.12 (Thesis GPU)' with PaddlePaddle 3.3+."
        )

    model_prefix = str(pretrained_dir / 'inference')
    model = paddle.jit.load(model_prefix)
    if for_training:
        model.train()
    else:
        model.eval()

    all_params = list(model.parameters())
    trainable_params = [p for p in all_params if not p.stop_gradient]
    if for_training and len(trainable_params) == 0:
        raise RuntimeError("Loaded recognition checkpoint has 0 trainable parameters (stop_gradient=True on all tensors).")

    # Probe output tensor shape & probability vs. logit activation with a tiny batch=1 tensor
    was_training = model.training
    model.eval()
    with paddle.no_grad():
        probe_x = paddle.zeros(
            [1, 3, PADDLEOCR_TRAINING_CONFIG['img_height'], PADDLEOCR_TRAINING_CONFIG['img_width']],
            dtype='float32',
        )
        probe_out = model(probe_x)
        if isinstance(probe_out, (tuple, list)):
            probe_out = probe_out[0]
        out_shape = list(probe_out.shape)
        if len(out_shape) != 3 or out_shape[2] != NUM_CTC_CLASSES:
            raise ValueError(f"Model output shape {out_shape} incompatible with CTC lexicon size {NUM_CTC_CLASSES}.")
        row_sum = float(probe_out[0, 0].sum().item())
        out_min = float(probe_out.min().item())
        outputs_probabilities = (out_min >= 0.0 and abs(row_sum - 1.0) < 1e-3)
    if was_training:
        model.train()

    meta = {
        'total_param_tensors': len(all_params),
        'trainable_param_tensors': len(trainable_params),
        'probe_output_shape': out_shape,
        'outputs_post_softmax_probs': outputs_probabilities,
    }
    return model, meta


def compute_ctc_log_probs(model_out: paddle.Tensor, outputs_probs: bool = True) -> paddle.Tensor:
    """Converts recognition head output [B, T, C] into time-major log-probabilities [T, B, C] for Warp-CTC."""
    if isinstance(model_out, (tuple, list)):
        model_out = model_out[0]
    if outputs_probs:
        # PP-OCRv6_medium_rec inference graph includes final Softmax; convert safely to log-probabilities
        log_probs = paddle.log(paddle.clip(model_out, min=1e-7, max=1.0))
    else:
        log_probs = F.log_softmax(model_out, axis=-1)
    return log_probs.transpose([1, 0, 2])


def evaluate_recognition_loader(model, data_loader, outputs_probs: bool = True, collect_predictions: bool = False):
    """Runs no-grad evaluation returning loss and all required OCR recognition metrics."""
    was_training = model.training
    model.eval()
    ctc_loss_fn = paddle.nn.CTCLoss(blank=0, reduction='mean')

    total_loss = 0.0
    total_samples = 0
    all_gts, all_preds, all_fnames, all_crop_ids = [], [], [], []

    with paddle.no_grad():
        for batch_imgs, batch_labels, batch_tgt_lens, gt_texts, fnames, crop_ids in data_loader:
            bsz = batch_imgs.shape[0]
            raw_out = model(batch_imgs)
            if isinstance(raw_out, (tuple, list)):
                raw_out = raw_out[0]

            log_probs_tbc = compute_ctc_log_probs(raw_out, outputs_probs=outputs_probs)
            in_lens = paddle.full(shape=[bsz], fill_value=log_probs_tbc.shape[0], dtype='int64')
            batch_loss = ctc_loss_fn(log_probs_tbc, batch_labels, in_lens, batch_tgt_lens)

            loss_val = float(batch_loss.item())
            if math.isfinite(loss_val):
                total_loss += loss_val * bsz
            total_samples += bsz

            decoded = ctc_greedy_decode(raw_out.numpy(), FULL_CHAR_LIST)
            all_gts.extend(gt_texts)
            all_preds.extend(decoded)
            if collect_predictions:
                all_fnames.extend(fnames)
                all_crop_ids.extend(crop_ids)

    if was_training:
        model.train()

    metrics = aggregate_ocr_metrics(all_gts, all_preds)
    metrics['loss'] = float(total_loss / max(1, total_samples))
    if collect_predictions:
        metrics['records'] = [
            {
                'crop_id': cid,
                'image_filename': fn,
                'ground_truth': gt,
                'predicted_text': pr,
                **compute_sample_ocr_metrics(gt, pr),
            }
            for cid, fn, gt, pr in zip(all_crop_ids, all_fnames, all_gts, all_preds)
        ]
    return metrics


# Verify split isolation (strictly check manifest IDs without loading test images or evaluating test data)
if TRAIN_MANIFEST.exists() and VAL_MANIFEST.exists() and TEST_MANIFEST.exists():
    _df_tr_meta = pd.read_csv(TRAIN_MANIFEST, usecols=['crop_id', 'document_id', 'image_path', 'image_filename', 'bounding_box', 'ground_truth_text', 'document_type'])
    _df_va_meta = pd.read_csv(VAL_MANIFEST, usecols=['crop_id', 'document_id', 'image_path', 'image_filename', 'bounding_box', 'ground_truth_text', 'document_type'])
    _df_te_ids_only = pd.read_csv(TEST_MANIFEST, usecols=['crop_id', 'document_id'])

    _tr_docs, _va_docs, _te_docs = set(_df_tr_meta['document_id']), set(_df_va_meta['document_id']), set(_df_te_ids_only['document_id'])
    assert _tr_docs.isdisjoint(_va_docs), "Data leakage detected between train and validation document_ids!"
    assert _tr_docs.isdisjoint(_te_docs), "Data leakage detected between train and test document_ids!"
    assert _va_docs.isdisjoint(_te_docs), "Data leakage detected between validation and test document_ids!"
    del _df_te_ids_only

    print(
        f"[SPLIT AUDIT] Verified zero document-level leakage across splits: "
        f"Train={len(_tr_docs)} docs ({len(_df_tr_meta):,} crops) | "
        f"Val={len(_va_docs)} docs ({len(_df_va_meta):,} crops) | "
        f"Held-Out Test={len(_te_docs)} docs (untouched)."
    )
else:
    _df_tr_meta = pd.DataFrame()
    _df_va_meta = pd.DataFrame()
    print("[WARNING] One or more split manifests are missing.")


### 5. Training Loop Implementation, Checkpoint Management & Safe Resume

**Checkpoint & Recovery Protocol:**
- Inspects `LAST_CKPT_DIR` (`model_weights.pdparams`, `optimizer.pdopt`, `scheduler.pdstates`, `checkpoint_meta.json`) and `HISTORY_CSV_PATH` before starting.
- Safely resumes from `last_completed_epoch + 1` if a valid prior checkpoint exists, preserving `best_val_cer` and `best_epoch` from `BEST_CKPT_DIR` so a superior checkpoint is **never** accidentally overwritten.
- Saves all `paddle.save(...)` artifacts using explicit `str(path)` string paths (avoiding `WindowsPath` serialization errors) via atomic temporary-file replacement.


In [ ]:
def atomic_paddle_save(state_obj, target_path: Path):
    """Writes Paddle state dict to temporary file first, then atomically replaces target_path."""
    target_path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = target_path.with_suffix(target_path.suffix + '.tmp')
    paddle.save(state_obj, str(tmp_path))
    os.replace(str(tmp_path), str(target_path))


def save_checkpoint_bundle(
    ckpt_dir: Path,
    model,
    optimizer,
    scheduler,
    metadata: dict,
    pretrained_dir: Path = PRETRAINED_REC_DIR,
    save_opt_and_sched: bool = True,
):
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    atomic_paddle_save(model.state_dict(), ckpt_dir / 'model_weights.pdparams')
    if save_opt_and_sched:
        if optimizer is not None:
            atomic_paddle_save(optimizer.state_dict(), ckpt_dir / 'optimizer.pdopt')
        if scheduler is not None:
            atomic_paddle_save(scheduler.state_dict(), ckpt_dir / 'scheduler.pdstates')

    for cfg_name in ('inference.yml', 'inference.json'):
        src_cfg = pretrained_dir / cfg_name
        dst_cfg = ckpt_dir / cfg_name
        if src_cfg.exists() and not dst_cfg.exists():
            shutil.copy2(str(src_cfg), str(dst_cfg))

    meta_tmp = ckpt_dir / 'checkpoint_meta.json.tmp'
    with open(meta_tmp, 'w', encoding='utf-8') as f:
        json.dump(metadata, f, indent=2)
    os.replace(str(meta_tmp), str(ckpt_dir / 'checkpoint_meta.json'))


def verify_saved_checkpoint(ckpt_dir: Path, require_optimizer: bool = False) -> tuple:
    """Verifies that saved checkpoint files exist on disk and are non-empty."""
    weights_file = ckpt_dir / 'model_weights.pdparams'
    meta_file = ckpt_dir / 'checkpoint_meta.json'
    if not (weights_file.exists() and weights_file.stat().st_size > 0):
        return False, None
    if require_optimizer:
        opt_file = ckpt_dir / 'optimizer.pdopt'
        sched_file = ckpt_dir / 'scheduler.pdstates'
        if not (opt_file.exists() and opt_file.stat().st_size > 0 and sched_file.exists() and sched_file.stat().st_size > 0):
            return False, None
    meta = {}
    if meta_file.exists() and meta_file.stat().st_size > 0:
        try:
            with open(meta_file, 'r', encoding='utf-8') as f:
                meta = json.load(f)
        except Exception:
            meta = {}
    return True, meta


# Instantiate Train & Validation Datasets (Test Manifest remains untouched)
df_train = _df_tr_meta if not _df_tr_meta.empty else (pd.read_csv(TRAIN_MANIFEST) if TRAIN_MANIFEST.exists() else pd.DataFrame())
df_val = _df_va_meta if not _df_va_meta.empty else (pd.read_csv(VAL_MANIFEST) if VAL_MANIFEST.exists() else pd.DataFrame())

train_dataset = OFPHILRecognitionDataset(df_train, split_name='train')
val_dataset = OFPHILRecognitionDataset(df_val, split_name='val')

print(f"Prepared Train Dataset : {len(train_dataset):,} valid crops (missing_imgs={train_dataset.missing_image_files}, ctc_truncated={train_dataset.truncated_for_ctc})")
print(f"Prepared Val Dataset   : {len(val_dataset):,} valid crops (missing_imgs={val_dataset.missing_image_files}, ctc_truncated={val_dataset.truncated_for_ctc})")

training_history = []

try:
    if len(train_dataset) == 0 or len(val_dataset) == 0:
        raise RuntimeError(
            "Train or Validation dataset has 0 resolvable image crops. Verify raw supporting document paths."
        )

    BATCH_SIZE = int(PADDLEOCR_TRAINING_CONFIG['batch_size'])
    EVAL_BATCH_SIZE = int(PADDLEOCR_TRAINING_CONFIG['eval_batch_size'])
    MAX_EPOCHS = int(PADDLEOCR_TRAINING_CONFIG['max_epochs'])
    BASE_LR = float(PADDLEOCR_TRAINING_CONFIG['learning_rate'])
    MIN_LR = float(PADDLEOCR_TRAINING_CONFIG['min_learning_rate'])
    WEIGHT_DECAY = float(PADDLEOCR_TRAINING_CONFIG['weight_decay'])
    GRAD_CLIP_NORM = float(PADDLEOCR_TRAINING_CONFIG['grad_clip_norm'])
    PATIENCE = int(PADDLEOCR_TRAINING_CONFIG['early_stopping_patience'])

    train_loader = paddle.io.DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        drop_last=False,
        num_workers=int(PADDLEOCR_TRAINING_CONFIG['num_workers']),
        collate_fn=ocr_ctc_collate_fn,
    )
    val_loader = paddle.io.DataLoader(
        val_dataset,
        batch_size=EVAL_BATCH_SIZE,
        shuffle=False,
        drop_last=False,
        num_workers=int(PADDLEOCR_TRAINING_CONFIG['num_workers']),
        collate_fn=ocr_ctc_collate_fn,
    )

    rec_model, model_meta = load_trainable_ppocrv6_rec(PRETRAINED_REC_DIR, for_training=True)
    OUTPUTS_PROBS = model_meta['outputs_post_softmax_probs']
    print(
        f"[MODEL VERIFIED] {PADDLEOCR_TRAINING_CONFIG['model_name']} | "
        f"Trainable Tensors: {model_meta['trainable_param_tensors']}/{model_meta['total_param_tensors']} | "
        f"Probe Output Shape: {model_meta['probe_output_shape']} | Post-Softmax Output: {OUTPUTS_PROBS}"
    )

    total_train_steps = max(1, MAX_EPOCHS * len(train_loader))
    lr_scheduler = paddle.optimizer.lr.CosineAnnealingDecay(
        learning_rate=BASE_LR,
        T_max=total_train_steps,
        eta_min=MIN_LR,
    )
    trainable_params = [p for p in rec_model.parameters() if not p.stop_gradient]
    grad_clip = paddle.nn.ClipGradByGlobalNorm(clip_norm=GRAD_CLIP_NORM)
    optimizer = paddle.optimizer.AdamW(
        learning_rate=lr_scheduler,
        parameters=trainable_params,
        weight_decay=WEIGHT_DECAY,
        grad_clip=grad_clip,
    )
    ctc_loss_fn = paddle.nn.CTCLoss(blank=0, reduction='mean')

    # Inspect Existing Checkpoints for Safe Recovery & Best-Checkpoint Protection
    start_epoch = 1
    best_val_cer = float('inf')
    best_val_ned = -1.0
    best_epoch = 0
    epochs_without_improvement = 0

    has_best_ckpt, best_meta = verify_saved_checkpoint(BEST_CKPT_DIR, require_optimizer=False)
    if has_best_ckpt and best_meta:
        best_val_cer = float(best_meta.get('val_cer', float('inf')))
        best_val_ned = float(best_meta.get('val_norm_ed', -1.0))
        best_epoch = int(best_meta.get('epoch', 0))
        print(f"[CHECKPOINT AUDIT] Existing BEST checkpoint detected at Epoch {best_epoch} (Val CER={best_val_cer:.4f}, 1-NED={best_val_ned:.4f}). Protected against worse overwrite.")

    has_last_ckpt, last_meta = verify_saved_checkpoint(LAST_CKPT_DIR, require_optimizer=True)
    if PADDLEOCR_TRAINING_CONFIG.get('resume_from_checkpoint', True) and has_last_ckpt and last_meta:
        completed_epoch = int(last_meta.get('epoch', 0))
        if completed_epoch > 0:
            rec_model.set_state_dict(paddle.load(str(LAST_CKPT_DIR / 'model_weights.pdparams')))
            lr_scheduler.set_state_dict(paddle.load(str(LAST_CKPT_DIR / 'scheduler.pdstates')))
            optimizer.set_state_dict(paddle.load(str(LAST_CKPT_DIR / 'optimizer.pdopt')))
            start_epoch = completed_epoch + 1
            epochs_without_improvement = int(last_meta.get('epochs_without_improvement', 0))
            if HISTORY_CSV_PATH.exists():
                df_prev_hist = pd.read_csv(HISTORY_CSV_PATH)
                training_history = [
                    row.to_dict() for _, row in df_prev_hist[df_prev_hist['epoch'] <= completed_epoch].iterrows()
                ]
            print(f"[RESUME] Restored model, AdamW state, and CosineAnnealing scheduler from Epoch {completed_epoch} (Next Epoch: {start_epoch}, LR: {lr_scheduler.get_lr():.6e}).")

    if start_epoch == 1 and not has_best_ckpt:
        print("\n[BASELINE CHECK] Computing pre-fine-tuning validation baseline (Epoch 0)...")
        base_metrics = evaluate_recognition_loader(rec_model, val_loader, outputs_probs=OUTPUTS_PROBS, collect_predictions=False)
        best_val_cer = base_metrics['cer']
        best_val_ned = base_metrics['norm_ed']
        print(
            f"--> Pretrained Baseline Val Loss: {base_metrics['loss']:.4f} | "
            f"Char Acc: {base_metrics['char_acc_pct']:.2f}% | CER: {base_metrics['cer']:.4f} | "
            f"WER: {base_metrics['wer']:.4f} | 1-NED: {base_metrics['norm_ed']:.4f} | "
            f"Exact Match: {base_metrics['exact_match_pct']:.2f}%"
        )

    if start_epoch > MAX_EPOCHS:
        PIPELINE_RUNTIME_STATUS = 'TRAINING COMPLETED'
        print(f"\n[STATUS] All {MAX_EPOCHS} configured epochs are already completed in {LAST_CKPT_DIR}.")
    else:
        PIPELINE_RUNTIME_STATUS = 'TRAINING IN PROGRESS'
        with open(TRAINING_STATE_JSON, 'w', encoding='utf-8') as f:
            json.dump({'status': PIPELINE_RUNTIME_STATUS, 'start_epoch': start_epoch, 'max_epochs': MAX_EPOCHS}, f, indent=2)

        print('\n' + '=' * 92)
        print(f"STARTING PP-OCRv6 SVTR RECOGNITION FINE-TUNING (EPOCHS {start_epoch}..{MAX_EPOCHS} | BATCH={BATCH_SIZE})")
        print('=' * 92)

        log_interval = max(1, len(train_loader) // 10)

        for epoch in range(start_epoch, MAX_EPOCHS + 1):
            t_start = time.time()
            rec_model.train()
            running_loss = 0.0
            seen_samples = 0

            for step, (batch_imgs, batch_labels, batch_tgt_lens, _, _, _) in enumerate(train_loader, 1):
                bsz = batch_imgs.shape[0]
                raw_out = rec_model(batch_imgs)
                log_probs_tbc = compute_ctc_log_probs(raw_out, outputs_probs=OUTPUTS_PROBS)
                in_lens = paddle.full(shape=[bsz], fill_value=log_probs_tbc.shape[0], dtype='int64')

                loss = ctc_loss_fn(log_probs_tbc, batch_labels, in_lens, batch_tgt_lens)
                loss_val = float(loss.item())
                if not math.isfinite(loss_val):
                    optimizer.clear_grad()
                    continue

                loss.backward()
                optimizer.step()
                optimizer.clear_grad()
                lr_scheduler.step()

                running_loss += loss_val * bsz
                seen_samples += bsz

                if step % log_interval == 0 or step == len(train_loader):
                    cur_lr = float(lr_scheduler.get_lr())
                    vram_now = get_gpu_vram_mb()
                    elapsed_s = time.time() - t_start
                    print(
                        f"Epoch [{epoch:02d}/{MAX_EPOCHS:02d}] | Step [{step:04d}/{len(train_loader):04d}] | "
                        f"Batch Loss: {loss_val:.4f} | LR: {cur_lr:.6e} | VRAM: {vram_now:.1f} MB | Elapsed: {elapsed_s:.1f}s"
                    )

            train_loss = float(running_loss / max(1, seen_samples))

            # Clear unused GPU cache prior to validation pass on 4 GB VRAM hardware
            if HAS_CUDA:
                paddle.device.cuda.empty_cache()
            gc.collect()

            val_metrics = evaluate_recognition_loader(rec_model, val_loader, outputs_probs=OUTPUTS_PROBS, collect_predictions=False)
            val_loss = val_metrics['loss']
            val_char_acc = val_metrics['char_acc_pct']
            val_cer = val_metrics['cer']
            val_wer = val_metrics['wer']
            val_ned = val_metrics['norm_ed']
            val_exact = val_metrics['exact_match_pct']
            epoch_lr = float(lr_scheduler.get_lr())
            duration_sec = float(time.time() - t_start)
            vram_peak_mb = get_gpu_vram_mb()

            # Check if validation CER strictly improved (or tied CER with higher 1-NED)
            improved = (val_cer < best_val_cer - 1e-6) or (abs(val_cer - best_val_cer) <= 1e-6 and val_ned > best_val_ned)
            if improved:
                best_val_cer = val_cer
                best_val_ned = val_ned
                best_epoch = epoch
                epochs_without_improvement = 0
                best_ckpt_meta = {
                    'epoch': epoch,
                    'train_loss': round(train_loss, 6),
                    'val_loss': round(val_loss, 6),
                    'val_char_acc_pct': round(val_char_acc, 4),
                    'val_cer': round(val_cer, 6),
                    'val_wer': round(val_wer, 6),
                    'val_norm_ed': round(val_ned, 6),
                    'val_exact_match_pct': round(val_exact, 4),
                    'learning_rate': epoch_lr,
                    'timestamp': time.strftime('%Y-%m-%dT%H:%M:%S'),
                }
                save_checkpoint_bundle(
                    BEST_CKPT_DIR,
                    rec_model,
                    optimizer,
                    lr_scheduler,
                    best_ckpt_meta,
                    save_opt_and_sched=True,
                )
            else:
                epochs_without_improvement += 1

            # Always persist latest recovery checkpoint + history at end of every completed epoch
            last_ckpt_meta = {
                'epoch': epoch,
                'best_epoch': best_epoch,
                'best_val_cer': round(best_val_cer, 6),
                'best_val_norm_ed': round(best_val_ned, 6),
                'epochs_without_improvement': epochs_without_improvement,
                'learning_rate': epoch_lr,
                'timestamp': time.strftime('%Y-%m-%dT%H:%M:%S'),
            }
            save_checkpoint_bundle(
                LAST_CKPT_DIR,
                rec_model,
                optimizer,
                lr_scheduler,
                last_ckpt_meta,
                save_opt_and_sched=True,
            )

            epoch_row = {
                'epoch': int(epoch),
                'train_loss': round(train_loss, 4),
                'val_loss': round(val_loss, 4),
                'val_char_acc_pct': round(val_char_acc, 2),
                'val_cer': round(val_cer, 4),
                'val_wer': round(val_wer, 4),
                'val_norm_ed': round(val_ned, 4),
                'val_exact_match_pct': round(val_exact, 2),
                'learning_rate': epoch_lr,
                'duration_sec': round(duration_sec, 2),
                'gpu_vram_mb': round(vram_peak_mb, 1),
                'is_best': bool(improved),
            }
            training_history.append(epoch_row)
            pd.DataFrame(training_history).to_csv(HISTORY_CSV_PATH, index=False)

            print('-' * 92)
            print(
                f"Epoch {epoch:02d}/{MAX_EPOCHS:02d} Summary | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
                f"Char Acc: {val_char_acc:.2f}% | CER: {val_cer:.4f} | WER: {val_wer:.4f} | "
                f"1-NED: {val_ned:.4f} | Exact Match: {val_exact:.2f}%"
            )
            print(
                f"LR: {epoch_lr:.6e} | Duration: {duration_sec:.1f}s | GPU VRAM: {vram_peak_mb:.1f} MB | "
                f"Best Epoch: {best_epoch} (Val CER: {best_val_cer:.4f})"
            )
            if improved:
                print(f"--> [SAVED BEST CHECKPOINT] Updated {BEST_CKPT_DIR}")
            print('-' * 92)

            if epochs_without_improvement >= PATIENCE:
                print(f"--> [EARLY STOPPING] Validation CER did not improve for {PATIENCE} consecutive epochs. Halting training.")
                break

        # Ensure a valid best checkpoint exists even if Epoch 0 baseline CER was not surpassed
        has_best_after, _ = verify_saved_checkpoint(BEST_CKPT_DIR, require_optimizer=False)
        if not has_best_after and verify_saved_checkpoint(LAST_CKPT_DIR, require_optimizer=False)[0]:
            print("[NOTICE] No trained epoch improved upon the Epoch 0 pretrained validation CER; preserving latest completed trained state in best_accuracy with explicit baseline comparison metadata.")
            for _fname in ('model_weights.pdparams', 'inference.yml', 'inference.json', 'checkpoint_meta.json'):
                _s = LAST_CKPT_DIR / _fname
                if _s.exists():
                    shutil.copy2(str(_s), str(BEST_CKPT_DIR / _fname))

        PIPELINE_RUNTIME_STATUS = 'TRAINING COMPLETED'
        with open(TRAINING_STATE_JSON, 'w', encoding='utf-8') as f:
            json.dump(
                {
                    'status': PIPELINE_RUNTIME_STATUS,
                    'completed_epochs': len(training_history),
                    'best_epoch': best_epoch,
                    'best_val_cer': best_val_cer,
                    'best_val_norm_ed': best_val_ned,
                    'history_csv': str(HISTORY_CSV_PATH),
                },
                f,
                indent=2,
            )
        print(f"\n[TRAINING COMPLETED] History saved -> {HISTORY_CSV_PATH}")

except Exception as exc:
    PIPELINE_RUNTIME_STATUS = 'FAILED – SEE ERROR'
    PIPELINE_ERROR_DETAIL = f"{type(exc).__name__}: {exc}"
    with open(TRAINING_STATE_JSON, 'w', encoding='utf-8') as f:
        json.dump({'status': PIPELINE_RUNTIME_STATUS, 'error': PIPELINE_ERROR_DETAIL}, f, indent=2)
    print(f"[FAILED – SEE ERROR] {PIPELINE_ERROR_DETAIL}")
    raise


### 6. Final Evaluation on Untouched Held-Out Test Partition (`test_manifest_v2.csv`)

Strictly loads the verified **best validation checkpoint** (`BEST_CKPT_DIR / 'model_weights.pdparams'`) after training has completed and evaluates it once on `test_manifest_v2.csv`. If no verified fine-tuned checkpoint exists yet, test evaluation is skipped truthfully without fabricating results.


In [ ]:
test_evaluation_summary = None
df_test_predictions = pd.DataFrame()

has_verified_best, best_ckpt_info = verify_saved_checkpoint(BEST_CKPT_DIR, require_optimizer=False)

if has_verified_best and TEST_MANIFEST.exists():
    try:
        df_test_manifest = pd.read_csv(TEST_MANIFEST)
        test_dataset = OFPHILRecognitionDataset(df_test_manifest, split_name='test')
        test_loader = paddle.io.DataLoader(
            test_dataset,
            batch_size=int(PADDLEOCR_TRAINING_CONFIG['eval_batch_size']),
            shuffle=False,
            drop_last=False,
            num_workers=int(PADDLEOCR_TRAINING_CONFIG['num_workers']),
            collate_fn=ocr_ctc_collate_fn,
        )

        eval_model, eval_meta = load_trainable_ppocrv6_rec(PRETRAINED_REC_DIR, for_training=False)
        eval_model.set_state_dict(paddle.load(str(BEST_CKPT_DIR / 'model_weights.pdparams')))
        eval_model.eval()

        print(
            f"[TEST EVALUATION] Loaded best validation checkpoint (Epoch {best_ckpt_info.get('epoch', 'N/A')}) "
            f"from {BEST_CKPT_DIR / 'model_weights.pdparams'}"
        )
        print(f"[TEST EVALUATION] Evaluating {len(test_dataset):,} untouched held-out test crops...")

        t_eval_start = time.time()
        test_res = evaluate_recognition_loader(
            eval_model,
            test_loader,
            outputs_probs=eval_meta['outputs_post_softmax_probs'],
            collect_predictions=True,
        )
        eval_elapsed_sec = time.time() - t_eval_start

        df_test_predictions = pd.DataFrame(test_res['records'])
        df_test_predictions.to_csv(TEST_PREDICTIONS_CSV, index=False, encoding='utf-8')

        test_evaluation_summary = {
            'model_name': PADDLEOCR_TRAINING_CONFIG['model_name'],
            'architecture': PADDLEOCR_TRAINING_CONFIG['architecture'],
            'detection_component_status': 'PP-OCRv6_medium_det (DBNet) — PRETRAINED / INFERENCE-ONLY',
            'checkpoint_evaluated': str(BEST_CKPT_DIR / 'model_weights.pdparams'),
            'selected_best_epoch': best_ckpt_info.get('epoch', None),
            'test_manifest': str(TEST_MANIFEST),
            'num_evaluated_samples': int(test_res['num_samples']),
            'test_loss': round(float(test_res['loss']), 4),
            'character_accuracy_pct': round(float(test_res['char_acc_pct']), 2),
            'cer': round(float(test_res['cer']), 4),
            'micro_cer': round(float(test_res['micro_cer']), 4),
            'wer': round(float(test_res['wer']), 4),
            'micro_wer': round(float(test_res['micro_wer']), 4),
            'normalized_edit_distance_1_minus_ned': round(float(test_res['norm_ed']), 4),
            'exact_match_accuracy_pct': round(float(test_res['exact_match_pct']), 2),
            'evaluation_duration_sec': round(float(eval_elapsed_sec), 2),
            'status': 'TEST EVALUATION COMPLETED',
        }

        with open(FINAL_REPORT_PATH, 'w', encoding='utf-8') as f:
            json.dump(test_evaluation_summary, f, indent=2)

        PIPELINE_RUNTIME_STATUS = 'TEST EVALUATION COMPLETED'
        if TRAINING_STATE_JSON.exists():
            with open(TRAINING_STATE_JSON, 'r', encoding='utf-8') as f:
                _st = json.load(f)
        else:
            _st = {}
        _st['status'] = PIPELINE_RUNTIME_STATUS
        _st['test_report_path'] = str(FINAL_REPORT_PATH)
        with open(TRAINING_STATE_JSON, 'w', encoding='utf-8') as f:
            json.dump(_st, f, indent=2)

        print(f"[TEST EVALUATION COMPLETED] Saved report -> {FINAL_REPORT_PATH}")
        print(f"[TEST PREDICTIONS SAVED]    Saved per-crop predictions -> {TEST_PREDICTIONS_CSV}")

    except Exception as exc:
        PIPELINE_RUNTIME_STATUS = 'FAILED – SEE ERROR'
        PIPELINE_ERROR_DETAIL = f"{type(exc).__name__}: {exc}"
        print(f"[FAILED – SEE ERROR] Held-out test evaluation error: {PIPELINE_ERROR_DETAIL}")
        raise
elif FINAL_REPORT_PATH.exists() and TEST_PREDICTIONS_CSV.exists():
    with open(FINAL_REPORT_PATH, 'r', encoding='utf-8') as f:
        test_evaluation_summary = json.load(f)
    df_test_predictions = pd.read_csv(TEST_PREDICTIONS_CSV)
    PIPELINE_RUNTIME_STATUS = test_evaluation_summary.get('status', 'TEST EVALUATION COMPLETED')
    print(f"Loaded previously completed test evaluation report from {FINAL_REPORT_PATH}.")
else:
    print("Status: PENDING MANUAL EXECUTION — No verified fine-tuned best checkpoint found yet. Run Section 5 first.")


### 7. Matplotlib Training Dynamics & Metric Trajectories

Generates the 6 required training curves strictly from actual recorded epoch history (`paddleocr_training_history.csv`):
1. Training Loss vs. Validation Loss
2. Character Accuracy (%) per Epoch
3. CER per Epoch
4. WER per Epoch
5. Exact Match Accuracy (%) per Epoch
6. Learning Rate per Epoch

*(No synthetic, target, or illustrative curves are ever plotted when training has not yet been executed.)*


In [ ]:
FIG_SAVE_PATH = OUTPUT_DIR / 'paddleocr_svtr_training_curves.png'

if HISTORY_CSV_PATH.exists() and HISTORY_CSV_PATH.stat().st_size > 0:
    df_h = pd.read_csv(HISTORY_CSV_PATH)
else:
    df_h = pd.DataFrame(training_history) if training_history else pd.DataFrame()

if not df_h.empty and 'epoch' in df_h.columns:
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    epochs_x = df_h['epoch']

    # 1. Training Loss vs. Validation Loss
    ax = axes[0, 0]
    ax.plot(epochs_x, df_h['train_loss'], marker='o', linewidth=2, color='#d9534f', label='Train Loss')
    ax.plot(epochs_x, df_h['val_loss'], marker='s', linewidth=2, linestyle='--', color='#337ab7', label='Validation Loss')
    ax.set_title('Training Loss vs. Validation Loss', fontsize=12, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('CTC Loss')
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.legend()

    # 2. Character Accuracy (%) per Epoch
    ax = axes[0, 1]
    ax.plot(epochs_x, df_h['val_char_acc_pct'], marker='o', linewidth=2, color='#2ca02c', label='Val Character Acc (%)')
    ax.set_title('Character Accuracy per Epoch', fontsize=12, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Character Accuracy (%)')
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.legend()

    # 3. CER per Epoch
    ax = axes[0, 2]
    ax.plot(epochs_x, df_h['val_cer'], marker='o', linewidth=2, color='#f0ad4e', label='Val CER')
    if 'val_norm_ed' in df_h.columns:
        ax.plot(epochs_x, df_h['val_norm_ed'], marker='d', linewidth=1.8, linestyle=':', color='#17becf', label='Val 1-NED')
    ax.set_title('CER (& 1-NED) per Epoch', fontsize=12, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Rate / Score')
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.legend()

    # 4. WER per Epoch
    ax = axes[1, 0]
    ax.plot(epochs_x, df_h['val_wer'], marker='^', linewidth=2, color='#9467bd', label='Val WER')
    ax.set_title('WER per Epoch', fontsize=12, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Word Error Rate (WER)')
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.legend()

    # 5. Exact Match Accuracy (%) per Epoch
    ax = axes[1, 1]
    ax.plot(epochs_x, df_h['val_exact_match_pct'], marker='o', linewidth=2, color='#1f77b4', label='Val Exact Match (%)')
    ax.set_title('Exact Match Accuracy per Epoch', fontsize=12, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Exact Match (%)')
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.legend()

    # 6. Learning Rate per Epoch
    ax = axes[1, 2]
    ax.plot(epochs_x, df_h['learning_rate'], marker='s', linewidth=2, color='#8c564b', label='Learning Rate')
    ax.set_title('Learning Rate Schedule per Epoch', fontsize=12, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Learning Rate')
    ax.ticklabel_format(axis='y', style='sci', scilimits=(0, 0))
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.legend()

    plt.suptitle('PP-OCRv6_medium_rec (SVTR) Fine-Tuning Dynamics on OFPHIL', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(FIG_SAVE_PATH, dpi=200, bbox_inches='tight')
    plt.show()
    print(f"Saved recorded training visualization -> {FIG_SAVE_PATH}")
else:
    print("No recorded training history found yet (Status: PENDING MANUAL EXECUTION). Skipping plot generation — no synthetic curves generated.")


### 8. Final Structured Summary Tables & Truthful Artifact Audit

Displays pandas tables (`display()`) for:
- **A. Training History**
- **B. Final Held-Out Test Evaluation**
- **C. Held-Out Test Prediction Samples**
- **D. Model Artifact Summary**
- **E. Final Training Status**


In [ ]:
# Resolve truthful final status from disk & runtime state
_has_best_disk, _best_meta_disk = verify_saved_checkpoint(BEST_CKPT_DIR, require_optimizer=False)
_has_last_disk, _last_meta_disk = verify_saved_checkpoint(LAST_CKPT_DIR, require_optimizer=True)
_has_hist_disk = HISTORY_CSV_PATH.exists() and HISTORY_CSV_PATH.stat().st_size > 0
_has_test_disk = FINAL_REPORT_PATH.exists() and FINAL_REPORT_PATH.stat().st_size > 0

if PIPELINE_ERROR_DETAIL is not None:
    FINAL_TRUTHFUL_STATUS = 'FAILED – SEE ERROR'
elif _has_test_disk or test_evaluation_summary is not None:
    FINAL_TRUTHFUL_STATUS = 'TEST EVALUATION COMPLETED'
elif PIPELINE_RUNTIME_STATUS == 'TRAINING IN PROGRESS':
    FINAL_TRUTHFUL_STATUS = 'TRAINING IN PROGRESS'
elif _has_best_disk and _has_hist_disk:
    FINAL_TRUTHFUL_STATUS = 'TRAINING COMPLETED'
else:
    FINAL_TRUTHFUL_STATUS = 'PENDING MANUAL EXECUTION'

# --- A. Training History Table ---
print("=== A. TRAINING HISTORY ===")
if _has_hist_disk:
    df_hist_out = pd.read_csv(HISTORY_CSV_PATH)
elif training_history:
    df_hist_out = pd.DataFrame(training_history)
else:
    df_hist_out = pd.DataFrame(columns=[
        'epoch', 'train_loss', 'val_loss', 'val_char_acc_pct', 'val_cer',
        'val_wer', 'val_norm_ed', 'val_exact_match_pct', 'learning_rate',
        'duration_sec', 'gpu_vram_mb', 'is_best'
    ])

if not df_hist_out.empty:
    df_hist_view = df_hist_out.rename(columns={
        'epoch': 'Epoch',
        'train_loss': 'Train Loss',
        'val_loss': 'Validation Loss',
        'val_char_acc_pct': 'Character Accuracy (%)',
        'val_cer': 'CER',
        'val_wer': 'WER',
        'val_norm_ed': '1-NED',
        'val_exact_match_pct': 'Exact Match Accuracy (%)',
        'learning_rate': 'Learning Rate',
        'duration_sec': 'Duration (s)',
        'gpu_vram_mb': 'GPU VRAM (MB)',
        'is_best': 'Best Checkpoint',
    })
    display(df_hist_view)
else:
    display(pd.DataFrame([{'Notice': 'No completed epochs recorded yet (PENDING MANUAL EXECUTION).'}]))

# --- B. Final Test Evaluation Table ---
print("\n=== B. FINAL HELD-OUT TEST EVALUATION ===")
if test_evaluation_summary is not None:
    df_test_eval_table = pd.DataFrame([{
        'Model Architecture': test_evaluation_summary.get('architecture'),
        'Selected Best Epoch': test_evaluation_summary.get('selected_best_epoch'),
        'Evaluated Test Samples': test_evaluation_summary.get('num_evaluated_samples'),
        'Test Loss': test_evaluation_summary.get('test_loss'),
        'Character Accuracy (%)': test_evaluation_summary.get('character_accuracy_pct'),
        'CER': test_evaluation_summary.get('cer'),
        'WER': test_evaluation_summary.get('wer'),
        'Normalized Edit Distance (1-NED)': test_evaluation_summary.get('normalized_edit_distance_1_minus_ned'),
        'Exact Match Accuracy (%)': test_evaluation_summary.get('exact_match_accuracy_pct'),
        'Evaluation Status': test_evaluation_summary.get('status'),
    }])
    display(df_test_eval_table)
else:
    display(pd.DataFrame([{
        'Model Architecture': PADDLEOCR_TRAINING_CONFIG['architecture'],
        'Evaluated Test Samples': 0,
        'Evaluation Status': FINAL_TRUTHFUL_STATUS,
        'Note': 'Held-out test set untouched; awaiting verified best validation checkpoint.',
    }]))

# --- C. Prediction Samples Table ---
print("\n=== C. HELD-OUT TEST PREDICTION SAMPLES ===")
if not df_test_predictions.empty:
    df_pred_samples = df_test_predictions[[
        'image_filename', 'ground_truth', 'predicted_text', 'cer', 'wer', 'exact_match_pct'
    ]].head(15).copy()
    df_pred_samples['Exact Match'] = df_pred_samples['exact_match_pct'].map(lambda v: 'YES' if float(v) >= 100.0 else 'NO')
    df_pred_samples = df_pred_samples.drop(columns=['exact_match_pct']).rename(columns={
        'image_filename': 'Image Filename',
        'ground_truth': 'Ground Truth',
        'predicted_text': 'Predicted Text',
        'cer': 'CER',
        'wer': 'WER',
    })
    df_pred_samples['CER'] = df_pred_samples['CER'].round(4)
    df_pred_samples['WER'] = df_pred_samples['WER'].round(4)
    display(df_pred_samples)
else:
    display(pd.DataFrame([{
        'Image Filename': 'N/A',
        'Ground Truth': 'N/A',
        'Predicted Text': 'N/A',
        'CER': None,
        'WER': None,
        'Exact Match': 'PENDING TEST EVALUATION',
    }]))

# --- D. Model Artifact Summary Table ---
print("\n=== D. MODEL ARTIFACT SUMMARY ===")
df_artifact_summary = pd.DataFrame([
    {
        'Artifact': 'Pretrained Recognition Checkpoint (SVTR)',
        'Path': str(PRETRAINED_REC_DIR),
        'Exists on Disk': PRETRAINED_REC_DIR.exists(),
        'Model Architecture': PADDLEOCR_TRAINING_CONFIG['architecture'],
        'Status': 'VERIFIED PRETRAINED BASELINE' if PRETRAINED_REC_DIR.exists() else 'MISSING',
    },
    {
        'Artifact': 'Pretrained Detection Checkpoint (DBNet)',
        'Path': str(PRETRAINED_DET_DIR),
        'Exists on Disk': PRETRAINED_DET_DIR.exists(),
        'Model Architecture': 'LCNetV4 + RepLKFPN + DBPostProcess (DBNet)',
        'Status': 'PRETRAINED / INFERENCE-ONLY (Not Fine-Tuned)',
    },
    {
        'Artifact': 'Fine-Tuned Best Validation Checkpoint',
        'Path': str(BEST_CKPT_DIR / 'model_weights.pdparams'),
        'Exists on Disk': _has_best_disk,
        'Model Architecture': PADDLEOCR_TRAINING_CONFIG['architecture'],
        'Status': f"SAVED (Epoch {_best_meta_disk.get('epoch', '?')})" if _has_best_disk else 'NOT YET GENERATED',
    },
    {
        'Artifact': 'Latest Recovery Checkpoint (+ Optimizer & Scheduler)',
        'Path': str(LAST_CKPT_DIR / 'model_weights.pdparams'),
        'Exists on Disk': _has_last_disk,
        'Model Architecture': PADDLEOCR_TRAINING_CONFIG['architecture'],
        'Status': f"SAVED (Epoch {_last_meta_disk.get('epoch', '?')})" if _has_last_disk else 'NOT YET GENERATED',
    },
    {
        'Artifact': 'Training History CSV',
        'Path': str(HISTORY_CSV_PATH),
        'Exists on Disk': _has_hist_disk,
        'Model Architecture': PADDLEOCR_TRAINING_CONFIG['architecture'],
        'Status': f'RECORDED ({len(df_hist_out)} epochs)' if _has_hist_disk else 'NOT YET GENERATED',
    },
    {
        'Artifact': 'Fine-Tuning Configuration YAML',
        'Path': str(CONFIG_YAML_PATH),
        'Exists on Disk': CONFIG_YAML_PATH.exists(),
        'Model Architecture': PADDLEOCR_TRAINING_CONFIG['architecture'],
        'Status': 'SAVED' if CONFIG_YAML_PATH.exists() else 'PENDING CELL EXECUTION',
    },
    {
        'Artifact': 'Held-Out Test Evaluation Report JSON',
        'Path': str(FINAL_REPORT_PATH),
        'Exists on Disk': _has_test_disk,
        'Model Architecture': PADDLEOCR_TRAINING_CONFIG['architecture'],
        'Status': FINAL_TRUTHFUL_STATUS,
    },
])
display(df_artifact_summary)

# --- E. Final Training Status ---
print("\n=== E. FINAL TRAINING STATUS ===")
df_final_status = pd.DataFrame([{
    'Target Recognition Model': PADDLEOCR_TRAINING_CONFIG['model_name'],
    'Recognition Architecture': PADDLEOCR_TRAINING_CONFIG['architecture'],
    'Detection Model Status': 'PP-OCRv6_medium_det (DBNet) — PRETRAINED / INFERENCE-ONLY',
    'Final Pipeline Status': FINAL_TRUTHFUL_STATUS,
    'Error Detail': PIPELINE_ERROR_DETAIL or 'None',
}])
display(df_final_status)


### Findings & Governance Notes
- **Architectural Separation:** Text detection (`PP-OCRv6_medium_det` / DBNet) is strictly documented as pretrained/inference-only; this notebook fine-tunes only text recognition (`PP-OCRv6_medium_rec` / LCNetV4 + EncoderWithLightSVTR) with CTC loss across its native 18,710-entry vocabulary.
- **Hardware & Kernel Notes:** Configured for NVIDIA RTX 3050 Laptop GPU (4 GB VRAM) at `batch_size=8` and `[3, 48, 640]` crops (`T=80` CTC time steps). Because `PP-OCRv6_medium_rec` is serialized as a Paddle PIR graph (`inference.json`), execute this notebook using the **PaddlePaddle >= 3.0 GPU kernel** (`venv312` / `Python 3.12 (Thesis GPU)`).
- **Strict Partition Isolation:** `test_manifest_v2.csv` is verified disjoint by `document_id` and is never accessed during training, learning rate scheduling, early stopping, or checkpoint selection.
